# ETL Temperature Data avec PySpark

Pipeline ETL complet pour analyser les données de température globale par ville avec Apache Spark

## 1. Configuration et Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, min, max, year, month, 
    when, isnan, isnull, round, lit, regexp_replace,
    to_date, desc, asc, row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, DateType
import time
import json
from datetime import datetime

print("="*70)
print("PIPELINE ETL - GLOBAL LAND TEMPERATURES BY CITY")
print("="*70)

## 2. Initialisation de Spark Session

In [ ]:
# Créer une session Spark avec configuration optimisée
spark = SparkSession.builder \
    .appName("ETL_Temperature_Analysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

# Configuration du niveau de log
spark.sparkContext.setLogLevel("WARN")

print("\n✓ Spark Session créée avec succès")
print(f"  Version Spark: {spark.version}")
print(f"  Application ID: {spark.sparkContext.applicationId}")
print(f"  UI disponible sur: http://localhost:4040")
print("\n" + "="*70)

## 3. Phase EXTRACT - Chargement des données

In [ ]:
print("\nPHASE 1: EXTRACTION DES DONNÉES")
print("-" * 70)

start_time = time.time()

# Chemin du fichier
file_path = r"C:\Users\dylan\OneDrive\Documents\CESI\COURS\Traitement des données - TP\GlobalLandTemperaturesByCity.csv"

# Lire le CSV avec Spark
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(file_path)

extract_time = time.time() - start_time

print(f"✓ Données chargées en {extract_time:.2f}s")
print(f"  Nombre de lignes: {df_raw.count():,}")
print(f"  Nombre de colonnes: {len(df_raw.columns)}")
print(f"  Partitions: {df_raw.rdd.getNumPartitions()}")

# Afficher le schéma
print("\nSchéma des données:")
df_raw.printSchema()

# Aperçu des données
print("\nAperçu des données:")
df_raw.show(5, truncate=False)

## 4. Phase TRANSFORM - Nettoyage et transformations

In [ ]:
print("\nPHASE 2: TRANSFORMATION DES DONNÉES")
print("-" * 70)

transform_start = time.time()

# Étape 1: Analyse des valeurs manquantes
print("\nÉtape 1: Analyse des valeurs manquantes")
null_counts = df_raw.select(
    [
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df_raw.columns
    ]
)
null_counts.show()

# Étape 2: Conversion de la colonne date
print("\nÉtape 2: Conversion des types de données")
df_transformed = df_raw \
    .withColumn("date", to_date(col("dt"), "yyyy-MM-dd")) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("temperature", col("AverageTemperature").cast(DoubleType())) \
    .withColumn("uncertainty", col("AverageTemperatureUncertainty").cast(DoubleType()))

# Étape 3: Nettoyage des données
print("\nÉtape 3: Nettoyage des valeurs nulles")
rows_before = df_transformed.count()

df_clean = df_transformed \
    .filter(col("temperature").isNotNull()) \
    .filter(col("City").isNotNull()) \
    .filter(col("Country").isNotNull()) \
    .filter(col("date").isNotNull())

rows_after = df_clean.count()

print(f"  Lignes avant nettoyage: {rows_before:,}")
print(f"  Lignes après nettoyage: {rows_after:,}")
print(f"  Lignes supprimées: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

# Étape 4: Sélectionner et renommer les colonnes finales
print("\nÉtape 4: Sélection des colonnes finales")
df_final = df_clean.select(
    col("date"),
    col("year"),
    col("month"),
    col("temperature"),
    col("uncertainty"),
    col("City").alias("city"),
    col("Country").alias("country"),
    col("Latitude").alias("latitude"),
    col("Longitude").alias("longitude")
)

# Cache pour optimiser les requêtes suivantes
df_final.cache()

transform_time = time.time() - transform_start
print(f"\n✓ Transformation complétée en {transform_time:.2f}s")

# Afficher le résultat
print("\nDonnées transformées:")
df_final.show(10, truncate=False)

## 5. Analyses et agrégations

In [ ]:
print("\nPHASE 3: ANALYSES ET AGRÉGATIONS")
print("-" * 70)

# Analyse 1: Température moyenne par pays
print("\nAnalyse 1: Top 10 des pays les plus chauds (température moyenne)")
temp_by_country = df_final.groupBy("country") \
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        round(avg("uncertainty"), 2).alias("avg_uncertainty"),
        count("*").alias("num_records")
    ) \
    .orderBy(desc("avg_temperature")) \
    .limit(10)

temp_by_country.show()

# Analyse 2: Température moyenne par année
print("\nAnalyse 2: Évolution de la température moyenne globale par année")
temp_by_year = df_final.groupBy("year") \
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        count("*").alias("num_records")
    ) \
    .orderBy("year")

# Afficher les 10 premières et dernières années
print("\nPremières années:")
temp_by_year.limit(10).show()

print("\nDernières années:")
temp_by_year.orderBy(desc("year")).limit(10).show()

# Analyse 3: Villes les plus chaudes et les plus froides
print("\nAnalyse 3: Top 10 villes les plus chaudes")
hottest_cities = df_final.groupBy("city", "country") \
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        count("*").alias("num_records")
    ) \
    .filter(col("num_records") > 100) \
    .orderBy(desc("avg_temperature")) \
    .limit(10)

hottest_cities.show(truncate=False)

print("\nAnalyse 4: Top 10 villes les plus froides")
coldest_cities = df_final.groupBy("city", "country") \
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        count("*").alias("num_records")
    ) \
    .filter(col("num_records") > 100) \
    .orderBy(asc("avg_temperature")) \
    .limit(10)

coldest_cities.show(truncate=False)

# Analyse 5: Température par mois (saisonnalité)
print("\nAnalyse 5: Température moyenne globale par mois (saisonnalité)")
temp_by_month = df_final.groupBy("month") \
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        count("*").alias("num_records")
    ) \
    .orderBy("month")

temp_by_month.show()

## 6. Détection d'anomalies - Températures extrêmes

In [ ]:
print("\nPHASE 4: DÉTECTION D'ANOMALIES")
print("-" * 70)

# Calculer les statistiques globales
stats = df_final.select(
    avg("temperature").alias("mean"),
    min("temperature").alias("min"),
    max("temperature").alias("max")
).first()

mean_temp = stats["mean"]
min_temp = stats["min"]
max_temp = stats["max"]

print(f"\nStatistiques globales:")
print(f"  Température moyenne: {mean_temp:.2f}°C")
print(f"  Température minimale: {min_temp:.2f}°C")
print(f"  Température maximale: {max_temp:.2f}°C")

# Températures extrêmes (< -20°C ou > 40°C)
print("\nDétection des températures extrêmes (< -20°C ou > 40°C):")
extreme_temps = df_final.filter(
    (col("temperature") < -20) | (col("temperature") > 40)
).select(
    "date", "city", "country", "temperature", "latitude", "longitude"
).orderBy(desc("temperature"))

print(f"\nNombre d'observations extrêmes: {extreme_temps.count():,}")
print("\nTop 10 températures les plus élevées:")
extreme_temps.limit(10).show(truncate=False)

print("\nTop 10 températures les plus basses:")
extreme_temps.orderBy(asc("temperature")).limit(10).show(truncate=False)

## 7. Analyse temporelle - Réchauffement climatique

In [ ]:
print("\nPHASE 5: ANALYSE DU RÉCHAUFFEMENT CLIMATIQUE")
print("-" * 70)

# Comparer les périodes 1900-1950 vs 1970-2020
print("\nComparaison des températures moyennes par période:")

period_comparison = df_final.groupBy(
    when((col("year") >= 1900) & (col("year") < 1950), "1900-1949")
    .when((col("year") >= 1950) & (col("year") < 2000), "1950-1999")
    .when(col("year") >= 2000, "2000+")
    .alias("period")
) \
.filter(col("period").isNotNull()) \
.agg(
    round(avg("temperature"), 2).alias("avg_temperature"),
    count("*").alias("num_records")
) \
.orderBy("period")

period_comparison.show()

# Tendance par décennie
print("\nTempérature moyenne par décennie:")
decade_trend = df_final \
    .withColumn("decade", (col("year") / 10).cast("int") * 10) \
    .groupBy("decade") \
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        count("*").alias("num_records")
    ) \
    .orderBy("decade")

decade_trend.show(30)

## 8. Phase LOAD - Sauvegarde des résultats

In [ ]:
print("\nPHASE 6: SAUVEGARDE DES RÉSULTATS")
print("-" * 70)

load_start = time.time()

output_dir = r"C:\Users\dylan\OneDrive\Documents\CESI\COURS\Traitement des données - TP\etl_output"

# 1. Sauvegarder les données nettoyées en Parquet (partitionné par année)
print("\n1. Sauvegarde des données nettoyées (Parquet partitionné)...")
df_final.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(f"{output_dir}/temperature_cleaned_parquet")
print("  ✓ Données nettoyées sauvegardées")

# 2. Sauvegarder températures par pays (CSV)
print("\n2. Sauvegarde température par pays (CSV)...")
temp_by_country.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{output_dir}/temperature_by_country")
print("  ✓ Températures par pays sauvegardées")

# 3. Sauvegarder évolution temporelle (CSV)
print("\n3. Sauvegarde évolution par année (CSV)...")
temp_by_year.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{output_dir}/temperature_by_year")
print("  ✓ Évolution annuelle sauvegardée")

# 4. Sauvegarder villes extrêmes (JSON)
print("\n4. Sauvegarde villes les plus chaudes (JSON)...")
hottest_cities.coalesce(1).write \
    .mode("overwrite") \
    .json(f"{output_dir}/hottest_cities")
print("  ✓ Villes les plus chaudes sauvegardées")

print("\n5. Sauvegarde villes les plus froides (JSON)...")
coldest_cities.coalesce(1).write \
    .mode("overwrite") \
    .json(f"{output_dir}/coldest_cities")
print("  ✓ Villes les plus froides sauvegardées")

# 5. Sauvegarder anomalies (Parquet)
print("\n6. Sauvegarde températures extrêmes (Parquet)...")
extreme_temps.write \
    .mode("overwrite") \
    .parquet(f"{output_dir}/extreme_temperatures")
print("  ✓ Températures extrêmes sauvegardées")

# 6. Sauvegarder tendance décennale (CSV)
print("\n7. Sauvegarde tendance par décennie (CSV)...")
decade_trend.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{output_dir}/decade_trend")
print("  ✓ Tendance décennale sauvegardée")

load_time = time.time() - load_start
print(f"\n✓ Sauvegarde complétée en {load_time:.2f}s")
print(f"\nRésultats disponibles dans: {output_dir}")

## 9. Génération du rapport final

In [ ]:
print("\nGÉNÉRATION DU RAPPORT FINAL")
print("="*70)

total_time = extract_time + transform_time + load_time

# Créer un rapport détaillé
report = {
    "pipeline_info": {
        "name": "ETL Global Land Temperatures",
        "execution_date": datetime.now().isoformat(),
        "spark_version": spark.version,
        "total_duration_seconds": round(total_time, 2)
    },
    "extract": {
        "duration_seconds": round(extract_time, 2),
        "rows_extracted": df_raw.count(),
        "columns": len(df_raw.columns),
        "partitions": df_raw.rdd.getNumPartitions()
    },
    "transform": {
        "duration_seconds": round(transform_time, 2),
        "rows_before_cleaning": rows_before,
        "rows_after_cleaning": rows_after,
        "rows_removed": rows_before - rows_after,
        "removal_percentage": round((rows_before - rows_after) / rows_before * 100, 2)
    },
    "load": {
        "duration_seconds": round(load_time, 2),
        "output_directory": output_dir,
        "formats": ["parquet", "csv", "json"],
        "datasets_saved": 7
    },
    "statistics": {
        "global_mean_temperature": round(mean_temp, 2),
        "global_min_temperature": round(min_temp, 2),
        "global_max_temperature": round(max_temp, 2),
        "extreme_observations": extreme_temps.count()
    }
}

# Sauvegarder le rapport en JSON
report_path = f"{output_dir}/etl_report.json"
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\n" + "="*70)
print("RÉSUMÉ DE L'EXÉCUTION")
print("="*70)
print(f"\n✓ Pipeline ETL terminé avec succès!")
print(f"\nTemps d'exécution:")
print(f"  - Extraction: {extract_time:.2f}s")
print(f"  - Transformation: {transform_time:.2f}s")
print(f"  - Chargement: {load_time:.2f}s")
print(f"  - TOTAL: {total_time:.2f}s")

print(f"\nStatistiques:")
print(f"  - Lignes traitées: {rows_after:,}")
print(f"  - Température moyenne globale: {mean_temp:.2f}°C")
print(f"  - Plage de température: {min_temp:.2f}°C à {max_temp:.2f}°C")
print(f"  - Observations extrêmes: {extreme_temps.count():,}")

print(f"\nRésultats sauvegardés dans: {output_dir}")
print(f"Rapport détaillé: {report_path}")
print(f"\nSpark UI: http://localhost:4040")
print("="*70)

## 10. Nettoyage et fermeture de la session Spark

In [ ]:
# Unpersist le cache
df_final.unpersist()

# Arrêter la session Spark
# spark.stop()
# print("\n✓ Session Spark arrêtée")

print("\n⚠️ Session Spark toujours active pour consultation dans Spark UI")
print("Exécutez 'spark.stop()' manuellement pour fermer la session")